#### **FastF1 References**

Installation: [https://docs.fastf1.dev/getting_started/installation.html](https://docs.fastf1.dev/getting_started/installation.html)</br>
API Documentation: [https://docs.fastf1.dev/api_reference/index.html](https://docs.fastf1.dev/api_reference/index.html)

In [ ]:
# Uncomment and run to install
# %pip install fastf1

In [ ]:
# Imports
import os

import fastf1 as ff1
import pandas as pd

from fastf1.req import RateLimitExceededError

#### **Event Data (FastF1)**

- Retrieve location data and start times for events from the years 2018-2025
- Only 'Race' data selected, exlcudes Qualifiers


In [ ]:
event_times = []

# Get race starting times and locations for each year
for year in range(2018, 2026):
    event_schedule = ff1.get_event_schedule(year, include_testing=False)
    
    # Loop through season schedule, storing race info
    for _, event in event_schedule.iterrows():
        for i in range(1, 6):
            if event[f'Session{i}'] == 'Race':
                race_start = event[f'Session{i}DateUtc']
                break
        
        event_times.append({
            'EventName': event['EventName'],
            'Country': event['Country'],
            'Location': event['Location'],
            'StartTime': race_start,
            'Year': year
        })
        
event_data = pd.DataFrame(event_times)
    

#### **Location Data**

- Data downloaded from a GitHub repository `f1-curcuits` created by bacinger
- Includes Longitude and Latitude of tracks from 1950 to present
- Location data will be used to aid in weather data collection for each track

Source: [https://github.com/bacinger/f1-circuits](https://github.com/bacinger/f1-circuits)<br>
Locations JSON: [https://github.com/bacinger/f1-circuits/blob/master/f1-locations.json](https://github.com/bacinger/f1-circuits/blob/master/f1-locations.json)

In [ ]:
# Import location data for f1 races
url = "https://raw.githubusercontent.com/bacinger/f1-circuits/master/f1-locations.json"

location_data = pd.read_json(url)[['location', 'lon', 'lat']]

# location_data.head()

In [ ]:
# Normalize location names across both data sources
location_data['location'] = location_data['location'].replace({
    'Montreal': 'Montréal',
    'Spa Francorchamps': 'Spa-Francorchamps',
    'Sao Paulo': 'São Paulo',
    'Nürburg': 'Nürburgring',
    'Scarperia e San Piero': 'Mugello'
})

event_data['Location'] = event_data['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

# Attach long, lat to each f1 race location
event_location_df = event_data.merge(location_data, left_on='Location', right_on='location', how='left')
event_location_df = event_location_df.drop(columns=['location'])

# Store location data
event_location_df.to_csv('../data/event_location_data.csv', index=False)

#### **Race Data (FastF1)**

Missing Data:
- The first two races of 2018 do not have telemetry data for drivers
- No racing data for Italy Grand Prix 2018

**Note:** May need to rerun multiple times to pull all data (keeps its spot by checking for existing files)<br>
**FastF Rate Limit:** `500 requests per hour`

In [ ]:
# Arguements for different types of session data
data_config = {
    'laps': dict(laps=True, weather=False, telemetry=False, messages=False),
    'weather': dict(laps=False, weather=True, telemetry=False, messages=False),
    'telemetry': dict(laps=False, weather=False, telemetry=True, messages=False)
}

In [ ]:
# Fetch race data from 2018-2025 for specified data types
def fetch_race_data(data_type):
    for year in range(2018, 2026):
        os.makedirs(f'../data/{year}/{data_type}', exist_ok=True) # create directory
        
        try:
            event_schedule = ff1.get_event_schedule(year, include_testing=False)
        except RateLimitExceededError:
            print('ERROR: Rate Limit Exceeded (500 calls/hr). Please wait before rerunning.')
            return False

        #  Retrieve race data per event
        for location in event_schedule['Location']:
            event = location.lower().replace(' ', '_')
            path = f'../data/{year}/{data_type}/{data_type}_{event}_{year}.csv'  
            
            # Skip if already saved
            if os.path.exists(path):
                print(f"Skipping {event} {year} — already saved")
                continue
            
            try:
                # Load session data with given config
                session = ff1.get_session(year, location, 'Race')
                session.load(**data_config[data_type])
                
                if data_type == 'laps':
                    session.laps.to_csv(path, index=False)
                elif data_type == 'weather':
                    session.weather_data.to_csv(path, index=False)
                elif data_type == 'telemetry':
                    # Fetch telemetry data for each driver
                    for driver in session.drivers:
                        try:
                            tel = session.laps.pick_drivers(driver).telemetry
                            tel.to_csv(f'../data/{year}/telemetry/telemetry_{driver}_{event}_{year}.csv', index=False)
                        except RateLimitExceededError:
                            print('ERROR: Rate Limit Exceeded (500 calls/hr). Please wait before rerunning.')
                            return False
                        except Exception as e:
                            print(f"ERROR: {e}")
            
            except RateLimitExceededError:
                print('ERROR: Rate Limit Exceeded (500 calls/hr). Please wait before rerunning.')
                return False
            
            except Exception as e: 
                print(f"ERROR: {e}")

In [ ]:
# Retrieve and store race data ('laps', 'weather', 'telemetry')
fetch_race_data('weather')